In [0]:
%sql

CREATE TABLE IF NOT EXISTS lakehouse.enterprise.iot_device_data (
    device_id STRING,
    timestamp TIMESTAMP,
    temperature DOUBLE,
    humidity DOUBLE,
    pressure DOUBLE
)
USING DELTA
-- LOCATION 's3://katolakehouse/enterprise/iot_device_data' -- not neccessary
COMMENT 'Gold layer - business-ready IoT device data';

INSERT INTO lakehouse.enterprise.iot_device_data
SELECT * FROM lakehouse.raw.iot_device_data;

-- DESCRIBE EXTENDED lakehouse.enterprise.iot_device_data;

In [0]:
%sql

select current

In [0]:
%sql
-- 1. COLUMN MASKING

-- Create a masking function
CREATE FUNCTION lakehouse.default.mask_device_id(device_id STRING)
RETURN CASE
    WHEN is_account_group_member('admin') THEN device_id
    ELSE CONCAT('****-', SUBSTRING(device_id, -4, 4))
END;

-- Apply the mask to the device_id column
ALTER TABLE lakehouse.enterprise.iot_device_data
ALTER COLUMN device_id SET MASK lakehouse.default.mask_device_id;

In [0]:
%sql

-- Create a masking function
CREATE FUNCTION lakehouse.default.mask_device_id(device_id STRING)
RETURN CASE
    WHEN is_account_group_member('admin') THEN device_id
    ELSE CONCAT('****-', SUBSTRING(device_id, -4, 4))
END;

-- Apply the mask to the device_id column
ALTER TABLE lakehouse.enterprise.iot_device_data
ALTER COLUMN device_id SET MASK lakehouse.default.mask_device_id;

SELECT * FROM lakehouse.enterprise.iot_device_data
LIMIT 10;

In [0]:
%sql

-- 2. ROW MASKING

-- Create a row filter function
CREATE FUNCTION lakehouse.default.filter_temperature(temp DOUBLE)
RETURN CASE
    WHEN is_account_group_member('admin') THEN TRUE
    ELSE temp < 30.0
END;

-- Apply the row filter to the table
ALTER TABLE lakehouse.enterprise.iot_device_data
SET ROW FILTER lakehouse.default.filter_temperature ON (temperature);

In [0]:
%sql

-- 3. CREATE MASKING VIEWS

CREATE OR REPLACE VIEW lakehouse.enterprise.iot_device_data_masked
AS
SELECT
    CASE
        WHEN is_account_group_member('admin') THEN device_id
        ELSE mask(device_id, 'X', 'x', 'n', '*')
    END AS device_id,
    timestamp,
    temperature,
    humidity,
    pressure
FROM lakehouse.enterprise.iot_device_data;

-- Grant access to the view
-- GRANT SELECT ON lakehouse.enterprise.iot_device_data_masked TO `analysts`;

In [0]:
%sql

DESCRIBE TABLE EXTENDED lakehouse.enterprise.iot_device_data;

In [0]:
%sql

-- 2. ROW MASKING

-- Create a row filter function
CREATE FUNCTION lakehouse.default.filter_temperature(temp DOUBLE)
RETURN CASE
    WHEN is_account_group_member('admin') THEN TRUE
    ELSE temp < 30.0
END;

-- Apply the row filter to the table
ALTER TABLE lakehouse.enterprise.iot_device_data
SET ROW FILTER lakehouse.default.filter_temperature ON (temperature);

SELECT
    *
FROM lakehouse.information_schema.row_filters;